In [6]:
import os
import geopandas as gpd
import numpy as np
import rasterio

In [7]:
def clip_images(img_path,boundary,out_dir, grid = 50):

    ## This function creates 50 x 50 grids and clips the raster image into those chunks

    os.makedirs(out_dir,exist_ok = True) # create output directory
    bounds = gpd.read_file(boundary) # read the boundary file
    
    with rasterio.open(img_path) as img: # Opening the raster file
        bounds = bounds.to_crs(img.crs)  # reprojecting the boundary crs to match the raster crs
        img_bounds = img.bounds   # extracted the bounds of the raster
        img_crs = img.crs         # extracted Crs of the raster file
        if img.nodata is not None:
            nodata = img.nodata    # create a nodata variable 
        else:
            nodata = 0
        
        xmin, ymin, xmax, ymax = img_bounds  # assigned xmin, ymin, xmax, ymax to the image bounds
        
        grid_cells = [                                   # create a list for the 50 by 50 grids
            box(x, y, x + grid, y + grid)
            for x in np.arange(xmin, xmax, grid)
            for y in np.arange(ymin, ymax, grid)
        ]                                            

        # create a geodataframe from the grids
        grid_gdf = gpd.GeoDataFrame({'geometry': grid_cells}, crs=img_crs)  # created a geodataframe to store the geometries of the grids

        base = os.path.splitext(os.path.basename(img_path))[0]  # Extracted the filename of the raster file

        saved = 0
        
        ## Clip the raster file to the grids
        for idx, row in  grid_gdf.iterrows():

            b_mask = row['geometry']

            if not bounds.intersects(b_mask).any():
                continue
                
            out_image, out_transform = mask(img, [mapping(b_mask)],crop =True, nodata=nodata)

            is_nodata = np.all(out_image == nodata, axis = 0)
            is_white = np.all(out_image == nodata, axis = 0)
            empty_pct = np.mean(is_nodata | is_white) * 100

            if empty_pct >= 98:
                continue

            profile = img.meta.copy()
            profile.update({
                    "driver": "GTiff",
                    "height": out_image.shape[1],
                    "width": out_image.shape[2],
                    "transform": out_transform
            })

            out_path = os.path.join(out_dir,f"{base}_tile_{idx}.tif")   ## Save clipped raster
            with rasterio.open(out_path,"w",**profile) as dst:
                dst.write(out_image)
            saved += 1

        print(f"Saved {saved} of {len(grid_gdf)} tiles")

In [ ]:
imagery = r""
boundary = r""
out_dir = r""

clip_images(imagery,boundary,out_dir)